In [1]:
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import nltk
import os
os.chdir('..')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ariel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ariel\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ariel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)

In [3]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
le = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining['genres'])

In [4]:
# Eliminar la columna 'ratings' del conjunto de datos de entrenamiento
dataTraining.drop(columns=['rating'], inplace=True)


# Limpiar y normalizar el texto
def preprocess_text(text):
    # Eliminar caracteres especiales y convertir a minúsculas
    text = re.sub(r'\W', ' ', text)
    text = text.lower()

    # Eliminar stopwords y lematizar
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text
#Creación de una nueva variable llamada plot_2 para unir las varibles "title" y "plot"
dataTraining['plot_2'] = dataTraining['title'] + ", " + dataTraining['plot']
dataTraining['plot_2'] = dataTraining['plot_2'].apply(preprocess_text)

In [5]:
X_train, X_test, y_train_genres, y_test_genres = train_test_split(dataTraining['plot_2'], y_genres, test_size=0.2, random_state=42)

In [6]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfid = vectorizer.fit_transform(X_train.tolist())
X_test_tfid = vectorizer.transform(X_test.tolist())

In [7]:
log_clf=OneVsRestClassifier(LogisticRegression(max_iter=100000))
log_clf.fit(X_train_tfid, y_train_genres)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=100000))

In [8]:
y_test_proba = log_clf.predict(X_test_tfid)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

np.float64(0.57625708562646)

In [9]:
joblib.dump(log_clf,'api_proyecto2/movies_log.pkl',compress=3)

['api_proyecto2/movies_log.pkl']

In [10]:
joblib.dump(vectorizer,'api_proyecto2/movies_vect.pkl',compress=3)

['api_proyecto2/movies_vect.pkl']

In [11]:
log_clf.predict_proba(X_train_tfid[0])

array([[0.03724524, 0.02344638, 0.01380853, 0.03363818, 0.80057192,
        0.05007087, 0.08852143, 0.26528211, 0.03266221, 0.03414632,
        0.00854567, 0.01779409, 0.03413545, 0.05202253, 0.0414714 ,
        0.02489028, 0.00094009, 0.29476222, 0.03994719, 0.01527948,
        0.16957019, 0.04729695, 0.01940517, 0.01605825]])

In [12]:
from api_proyecto2.model_deployment import predict_api

predict_api('a','b')

{'p_Action': np.float64(0.11057630489517642),
 'p_Adventure': np.float64(0.08398604275565205),
 'p_Animation': np.float64(0.03126708213033806),
 'p_Biography': np.float64(0.03928666037963866),
 'p_Comedy': np.float64(0.4092609816964646),
 'p_Crime': np.float64(0.12044114665940231),
 'p_Documentary': np.float64(0.06018052857078376),
 'p_Drama': np.float64(0.3958286000749238),
 'p_Family': np.float64(0.066671687849001),
 'p_Fantasy': np.float64(0.058725275601808936),
 'p_Film-Noir': np.float64(0.013133252250855253),
 'p_History': np.float64(0.0325051939160094),
 'p_Horror': np.float64(0.08312459475554032),
 'p_Music': np.float64(0.03444458283172965),
 'p_Musical': np.float64(0.026324959733090087),
 'p_Mystery': np.float64(0.06554807500415895),
 'p_News': np.float64(0.0009256992726906156),
 'p_Romance': np.float64(0.14330680782601782),
 'p_Sci-Fi': np.float64(0.06023519065127793),
 'p_Short': np.float64(0.018046998007616912),
 'p_Sport': np.float64(0.0241306593205582),
 'p_Thriller': np.f

In [13]:
from flask import Flask
from flask_restx import Api, Resource, fields

In [19]:
app = Flask(__name__)

# Definición API Flask
api = Api(
    app, 
    version='1.0', 
    title='Movie Gender Classification API',
    description='Movie Gender Classification  API')

ns = api.namespace('predict', 
     description='Gender Classificator')

# Definición argumentos o parámetros de la API
parser = ns.parser()
parser.add_argument(
    'title', 
    type=str, 
    required=True, 
    help='Title of the movie', 
    location='args')

parser.add_argument(
    'plot', 
    type=str, 
    required=True, 
    help='Plot of the movie', 
    location='args')


resource_fields = api.model('Resource', {
    'result': fields.Raw,
})

In [20]:
# Definición de la clase para disponibilización
@ns.route('/')
class PopularityApi(Resource):

    @ns.doc(parser=parser)
    @ns.marshal_with(resource_fields)
    def get(self):
        args = parser.parse_args()
        
        return {
         "result": predict_api(args['title'],args['plot'])
        }, 200

In [21]:
app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.107:5000
Press CTRL+C to quit
192.168.0.107 - - [25/May/2025 12:25:15] "GET / HTTP/1.1" 200 -
192.168.0.107 - - [25/May/2025 12:25:15] "GET /swaggerui/droid-sans.css HTTP/1.1" 304 -
192.168.0.107 - - [25/May/2025 12:25:15] "GET /swaggerui/swagger-ui-bundle.js HTTP/1.1" 304 -
192.168.0.107 - - [25/May/2025 12:25:15] "GET /swaggerui/swagger-ui.css HTTP/1.1" 304 -
192.168.0.107 - - [25/May/2025 12:25:15] "GET /swaggerui/swagger-ui-standalone-preset.js HTTP/1.1" 304 -
192.168.0.107 - - [25/May/2025 12:25:16] "GET /swagger.json HTTP/1.1" 200 -
192.168.0.107 - - [25/May/2025 12:25:33] "GET /predict/?title='hi'&plot='hello' HTTP/1.1" 200 -
192.168.0.107 - - [25/May/2025 12:26:39] "GET /predict/?title='hi'&plot='hello' HTTP/1.1" 200 -
192.168.0.107 - - [25/May/2025 12:26:40] "GET /favicon.ico HTTP/1.1" 404 -
